# Objectives:

The main objective of this notebook is to explore the raw data structurally and obtain the information needed for data cleaning.
Please note that any changes in the dataset in this notebook are purely for exploration and will be neglected in the next notebooks.
- What are the data types of each column?
- Are there missing or "unknown" values? If there are, how much of the data do they represent?
- Is there a pattern behind missing values or are they random?
- Are there incorrect values such as negative values in features that should realistically contain only positive values?
- Could some special cases in the data contain hidden meanings?

# What we already know

About Dataset:\
The data contains transactions from 01/12/2010 to 09/12/2011 of a UK-based non-store online retail which sells mainly unique all-occasion gifts.\
Many customers of the company are wholesalers.

About Attributes:
- InvoiceNo: Invoice number. Nominal. A 6-digit integral number uniquely assigned to each order. Codes starting with "c" indicate a cancellation
- StockCode: Product code. Nominal. A 5-digit integral number uniquely assigned to each product.
- Description: Product name. Nominal.
- Quantity: Number of items bought per transaction. Numeric.
- InvoiceDate: Date and time of the generation of each transaction. Ranged from 01/12/2010 to 09/12/2011.
- UnitPrice: Unit price. Numeric. Product price per unit (in sterling).
- CustomerID: Customer ID. Nominal. A 5-digit integral number assigned uniquely to each customer.
- Country: Country name. Nominal. The country where each customer resides.

# Code:

## Initializing

In [1]:
import os
os.chdir('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
import datetime

from src.transformers import clear_stocks

In [2]:
raw_df = pd.read_csv('data/raw/Online_Retail.csv', encoding='ISO-8859-1')

In [3]:
# This copy is made just in case we want to reset df without loading the dataset again
df = raw_df.copy()

## Basic exploration

In [4]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/11 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/11 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/11 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/11 12:50,4.15,12680.0,France


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


The column InvoiceDate contains data of string type and not datetime64.

In [6]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [7]:
len(df) - len(df.drop_duplicates())

5268

Findings & Interpretations:\
Dataset & Data types:\
The dataset contains 541909 transactions (rows) and 8 starting features (columns).\
The dataset contains 5268 duplicated rows.\
The InvoiceNo column has data type string. This is because some invoice numbers start with the letter "c" to identify cancellations.\
The StockCode column has data type string. This could mean that stock codes contain non-numeric characters.\
The InvoiceDate column has data type string instead of datetime64.\
The time in the InvoiceDate column is a 24-hour cycle.\
The CustomerID column has data type float. This is weird since ID data are usually integer or string, but it could be because of the missing values.

Missing Data:\
The only columns to contain missing values are Description and CustomerID.\
Even though CustomerID has a lot of missing values, Country column has no missing values, which probably means that the source of the Country column is the order address and not the customer's profile.

Numbers & Percentages:\
The minimum and maximum values of the Quantity column happen to be the same, only differing in sign.\
The Quantity column's minimum value is negative. If multiple values in the Quantity column are negative too, that might mean returned items or cancellations.\
The UnitPrice column's minimum value is negative. If this occurs a lot then it could mean returned items or refund adjustments.\
The Quantity and UnitPrice columns' 75th percentile is much smaller than the maximum value. This suggests significant right-skewness, and extreme outliers in both columns.

Actions & Plans:\
Investigate the duplicated rows visually to see if they are truly duplicate and decide how to deal with them.\
Investigate further the InvoiceNo column to see if it contains only numeric characters (except for the "C" at the start of cancelled orders' invoices)\
The StockCode column must be investigated to see if product codes actually contain non-numeric characters, what are their patterns, and how to deal with them.\
Turn InvoiceDate to datetime64 to allow flawless extraction of features and help in exploration.\
Investigate the non-missing data in CustomerID column to make sure it doesn't contain decimal values.

Missing descriptions and customer IDs should be investigated and see if there are patterns behind them.\
Investigate if missing descriptions also mean incorrect or ambigiuous stock code.\
Ensure that the Country column contains only countries and with consistent labelling.

Investigate the meaning behind the minimum and maximum values of the Quantity having the same number but with opposite sign.\
Investigate further the negative values in the Quantity and UnitPrice columns and whether or not they are related to cancellations.\
Investigate the big gap in the Quantity and UnitPrice columns. (during EDA, not in this notebook)

## Duplicates Handling

How many invoices contain duplicated transaction rows?

In [8]:
# Creating a variable with the number of unique values of the "InvoiceNo" column that is in the duplicated rows of the dataframe
duplicate_df = df[df.duplicated()]
duplicate_invoices = duplicate_df['InvoiceNo'].nunique()
print('Invoices with duplicates:', duplicate_invoices)
print('Proportion of invoices with duplicates:', duplicate_invoices / df['InvoiceNo'].nunique())

Invoices with duplicates: 1933
Proportion of invoices with duplicates: 0.07463320463320464


~7% of invoices (1933 invoices) contain at least one duplicate transaction row.

In [9]:
print('Total duplicated rows:', len(duplicate_df))
print('Proportion of duplicated rows:', len(duplicate_df) / len(df))

Total duplicated rows: 5268
Proportion of duplicated rows: 0.009721189350979592


Duplicate rows represent the transactions where all records are identical.\
While it is theoritically possible for a customer to repurchase the same item with the same quantity in the same invoice at the exact same timestamp, the probability of this occuring is extremely low for it to happen in 5268 transactions (~0.1 of the dataset).\
It is more likely that these are system-generated errors or data-entry errors rather than genuine repeated transactions.\
Therefore, the duplicate rows will be removed.\
Although duplicate removal is typically part of the data cleaning phase, they will be temporarily removed in this notebook to properly explore the dataset statistically and structurally without being affected by the duplicate rows.

In [10]:
df = df.drop_duplicates()

In [11]:
len(df)

536641

The remaining rows of the dataframe after duplicate removal are 536,641 rows.

---

## Understanding the data

### Invoice Timestamps

The InvoiceDate column is stored as a string data type.
Even though the transformation to datetime64 type is typically done in the data cleaning process, it should be done temporarily in this notebook for exploratory feature engineering and better structural understanding of the dataset.

In [12]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceDate']

C:\Users\user\AppData\Local\Temp\ipykernel_16044\1563363543.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


0        2010-12-01 08:26:00
1        2010-12-01 08:26:00
2        2010-12-01 08:26:00
3        2010-12-01 08:26:00
4        2010-12-01 08:26:00
                 ...        
541904   2011-12-09 12:50:00
541905   2011-12-09 12:50:00
541906   2011-12-09 12:50:00
541907   2011-12-09 12:50:00
541908   2011-12-09 12:50:00
Name: InvoiceDate, Length: 536641, dtype: datetime64[us]

Let's create some temporary exploratory features from the timestamps to help explore the data.

In [13]:
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['Hour'] = df['InvoiceDate'].dt.hour

In [14]:
df['Year'].value_counts()

Year
2011    494660
2010     41981
Name: count, dtype: int64

In [15]:
print('Proportion of transactions in 2011:', len(df['Year'][df['Year'] == 2011]) / len(df))

Proportion of transactions in 2011: 0.9217707927646229


~92% of the transactions are in the year 2011.\
\~8% of the transactions are in the year 2010.\
This big difference makes sense since the transactions start in December 2010 (the end of the year 2010).

---

### Unique Values Inspection

Let's inspect the unique values in the dataset.

In [16]:
len(df)

536641

In [17]:
df.nunique()

InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
Year               2
Month             12
Day               31
Hour              15
dtype: int64

The length of the dataset is greater than the unique values of InvoiceNo, meaning that:
1. Some transactions may have the same invoice number.
2. Rows don't describe a whole order, but instead an item in an order.
3. Some orders may have multiple items.

The length of the dataset is greater than the unique values of InvoiceDate, meaning that some transactions may have the same timestamp.\
Unique values of the InvoiceDate column are less than unique values of InvoiceNo, meaning that multiple orders can happen at the same time.

Even though some of the Description column has missing values, the number of unique descriptions is greater than the number of unique stock codes. This means that some stock codes can point to more than one description, which is likely a data entry error.

There are only 4372 unique customer ids, but we can't fairly compare between this number and the number of unique invoice numbers since the CustomerID column contains missing values.\
Let's get the numbers of unique invoice numbers that don't have a missing customer id.

In [18]:
df[~df['CustomerID'].isna()].nunique()

InvoiceNo      22190
StockCode       3684
Description     3896
Quantity         436
InvoiceDate    20460
UnitPrice        620
CustomerID      4372
Country           37
Year               2
Month             12
Day               31
Hour              15
dtype: int64

The number of unique invoice numbers in rows that contain a customer id is still larger than the number of unique customer ids.

---

### Invoice inspection

Let's check whether invoices have more than one timestamp:

In [19]:
# Grouping by InvoiceNo to get the unique number of timestamps per invoice
invoice_timestamps_count = df.groupby('InvoiceNo')['InvoiceDate'].nunique()

# Returning the value counts of different timestamps per invoice
invoice_multiple_timestamps = invoice_timestamps_count[invoice_timestamps_count != 1]
invoice_multiple_timestamps.value_counts()

InvoiceDate
2    43
Name: count, dtype: int64

It seems that there are 43 orders that contain transactions with two different timestamps. Meaning that not all transactions that are within the same invoice happen at the same time.

How much is the difference in time? Are they only minutes away? Let's see:

In [20]:
# Grouping invoice dates by invoice numbers then taking the difference between each timestamp and the one before and summing all differences
df.groupby('InvoiceNo')['InvoiceDate'].diff().sum()

Timedelta('0 days 00:43:00')

The sum of all same-invoice timestamps differences is 43 minutes, which is the same amount of invoices with different timestamps. Meaning that whenever there is a difference in the timestamps within the same invoice, it is only 1 minute.

Now let's see if either of InvoiceNo and InvoiceDate are monotonic increasing:

In [21]:
df['InvoiceDate'].is_monotonic_increasing

True

InvoiceDate is monotonic increasing

Since the column InvoiceNo includes alphabetic characters, we must strip it of them first to check whether it's monotonic increasing.

In [22]:
# Creating a series containing invoice numbers without the alphabetic characters (if there are any)
invoices_stripped = df['InvoiceNo'].apply(lambda x: int(x[1:]) if x[0].isalpha() else int(x))

# Getting the unique values to check if invoices are monotonic increasing
invoices_stripped_unique = pd.Series(invoices_stripped.unique()) 
invoices_stripped_unique.is_monotonic_increasing

False

InvoiceNo is NOT monotonic increasing

In [23]:
invoices_stripped_unique.is_monotonic_decreasing

False

InvoiceNo is not monotonic decreasing either.\
Therefore InvoiceNo column is not sorted in any way. Only InvoiceDate is.\
So the dataset is sorted by date and time.

Let's see if all invoices are done by only one customer each:

In [24]:
# Getting the number of unique customers per invoice (sorted descending)
df.groupby('InvoiceNo')['CustomerID'].nunique().sort_values(ascending=False)

InvoiceNo
C581569    1
536365     1
536366     1
536367     1
536368     1
          ..
581435     0
581431     0
581498     0
581497     0
581492     0
Name: CustomerID, Length: 25900, dtype: int64

All invoices are done by only one customer. Except for the invoices having a missing customer id, of course.

---

### Cancelled Orders

It's already known that InvoiceNo may contain the letter "C" at the start of an invoice number to indicate a cancelled order, so let's check if there are other alphabetic characters present in invoice numbers.

In [25]:
# Applying a lambda function to the InvoiceNo column that returns the value counts of only the first character of every invoice number that contains non-numeric characters
df['InvoiceNo'].apply(lambda x: x[0] if not x.isdigit() else None).dropna().value_counts()

InvoiceNo
C    9251
A       3
Name: count, dtype: int64

There seems to be 3 invoice numbers starting with the letter "A". Let's check their rows in the dataset.

In [26]:
#Checking the rows in the df where the invoice number starts with an "A"
df[df['InvoiceNo'].isin(df['InvoiceNo'].apply(lambda x: x if x[0] == 'A' else None).dropna())]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day,Hour
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,2011,8,12,14
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,2011,8,12,14
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,2011,8,12,14


These transactions likely represent accounting adjustments and are not real transactions.\
Let's see if these adjustments only occur in rows with invoice numbers starting with "A":

In [27]:
df[df['Description'] == 'Adjust bad debt']

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day,Hour
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,2011,8,12,14
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,2011,8,12,14
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,2011,8,12,14


In [28]:
df[df['StockCode'] == 'B']

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day,Hour
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,2011,8,12,14
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,2011,8,12,14
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,2011,8,12,14


Yes, they are.\
Since they are only three, they can be considered as outliers and removed later on.

Let's see if cancelled invoice numbers are repeated without the "C":

In [29]:
# Creating a Series with only cancelled invoice numbers
cancelled_invoices = df['InvoiceNo'].apply(lambda x: x if x[0] == 'C' else None).dropna()

# Removing the "C" at the start of every cancelled invoice number
stripped_cancelled_invoices = cancelled_invoices.apply(lambda x: x[1:])

# Checking if the stripped_cancelled_invoices Series and InvoiceNo column have any elements in common
(stripped_cancelled_invoices.isin(df['InvoiceNo'])).any()

np.False_

Cancelled invoice numbers don't repeat without the "C".

---

### Country

Let's take a look on the present countries in the dataset:

In [30]:
df['Country'].value_counts()

Country
United Kingdom          490300
Germany                   9480
France                    8541
EIRE                      8184
Spain                     2528
Netherlands               2371
Belgium                   2069
Switzerland               1994
Portugal                  1510
Australia                 1258
Norway                    1086
Italy                      803
Channel Islands            757
Finland                    695
Cyprus                     611
Sweden                     461
Unspecified                442
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     294
USA                        291
Hong Kong                  284
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         58


There seems to be 442 "unspecified" countries.

In [31]:
# Creating a dataframe where the Country column is "Unspecified"
df_unspecified_country = df[df['Country'] == 'Unspecified']
print('Proportion of unspecified countries:', len(df_unspecified_country) / len(df))

Proportion of unspecified countries: 0.0008236418760400342


0.08% of the transactions have an unspecified country (446 rows out of 536,641).

Furthermore, let's see if transactions with an unspecified country have anything to do with customer id:

In [32]:
df_unspecified_country['CustomerID'].value_counts()

CustomerID
12743.0    131
16320.0     56
14265.0     31
12363.0     23
Name: count, dtype: int64

Transactions with an unspecified country seem to be done by only 4 customers.

In [33]:
df_unspecified_country['CustomerID'].value_counts().sum()

np.int64(241)

When summing up the numbers of appearances of all those customers (241), they are lesser than the total number of transactions with an unspecified country (422), meaning that the transactions with an unspecified country contain missing customer ids.

Let's see the invoices of transactions with an unspecified country:

In [34]:
df_unspecified_country['InvoiceNo'].value_counts()

InvoiceNo
561658    82
559521    72
565303    65
561661    49
552695    47
578539    34
576646    19
549687    16
564051    16
559929    15
553857    11
557499     9
563947     7
Name: count, dtype: int64

The unique invoice numbers in transactions with an unspecified country are too many for there to be a clear pattern.

What about stock codes?

In [35]:
df_unspecified_country['StockCode'].value_counts()

StockCode
22150    4
21124    3
22619    3
21888    3
22620    3
        ..
23507    1
21914    1
22560    1
23570    1
23571    1
Name: count, Length: 344, dtype: int64

Again, no pattern.

However, since rows with an unspecified country only represents 0.08% of the dataframe, it can be removed during data cleaning.

---

### Stock Codes

Now, let's see what are the most common stock codes across multiple invoices

In [36]:
df.groupby('StockCode')['InvoiceNo'].nunique().sort_values(ascending=False).head(10)

StockCode
85123A    2246
22423     2172
85099B    2135
47566     1706
20725     1608
84879     1468
22720     1462
22197     1442
21212     1334
22383     1306
Name: InvoiceNo, dtype: int64

There seems to be no pattern in the most common stock codes across multiple invoices.

#### Alphanumeric Stock Codes

The column StockCode is stored as a string data type so let's see if it contains non-digit characters:

In [37]:
# Creating a series with only alphanumeric stock codes
alphanumeric_stockcodes = df['StockCode'].apply(lambda x: x if not x.isdigit() else None).dropna()
print('Rows with alphanumeric stock codes:', len(alphanumeric_stockcodes))
print('Proportion of alphanumeric stock codes:', len(alphanumeric_stockcodes) / len(df))

Rows with alphanumeric stock codes: 54487
Proportion of alphanumeric stock codes: 0.10153342737509806


~10% of stock codes contain alphabetic characters, making stock codes alphanumeric data (containing both numeric and alphabetic characters).\
But that shouldn't matter since stock codes are not usually treated as numeric data anyway.\
However, the alphabetic characters may have meaning. Let's try to decode that meaning.

Throughout exploring so far, we've seen that some stock codes have alphabetic characters at the end of the code, and some may consist of only alphabetic characters.\
Let's explore stock codes with alphabetic characters at the end:

In [38]:
# Creating a series containing only alphanumeric stock codes with an alphabetic suffix
alpha_suffixes_stockcodes = alphanumeric_stockcodes.apply(lambda x: x if not (x[-1].isdigit() or x.isalpha()) else None).dropna()
print('Rows with alphanumeric stock codes with an alphabetic suffix:', len(alpha_suffixes_stockcodes))
print('Proportion of alphanumeric stock codes with an alphabetic suffix:', len(alpha_suffixes_stockcodes) / len(alphanumeric_stockcodes))
print('Proportion of stock codes with an alphabetic suffix:', len(alpha_suffixes_stockcodes) / len(df))

Rows with alphanumeric stock codes with an alphabetic suffix: 51536
Proportion of alphanumeric stock codes with an alphabetic suffix: 0.945840292179786
Proportion of stock codes with an alphabetic suffix: 0.09603440661447783


~95% of alphanumeric stock codes have an alphabetic suffix. Which represents ~10% of the total data.

Let's see what are the alphabetic characters that could be the suffix of stock codes:

In [39]:
alpha_suffixes_stockcodes.apply(lambda x: x[-1]).value_counts()

StockCode
B    13705
A    12590
C     6425
D     3601
L     2593
E     2169
F     1956
S     1687
G     1478
P      758
N      609
a      542
M      451
K      427
b      412
H      378
c      231
J      222
W      186
d      180
l      175
U      166
e      100
s       93
R       91
n       84
g       60
V       49
f       29
p       26
T       17
I       13
Z       10
k       10
Y        8
O        5
Name: count, dtype: int64

There seems to be lots of possibilities and there are even lower-case and upper-case letters. Let's try to interpret stock codes ending with "A".

In [40]:
df[df['StockCode'].apply(lambda x: x[-1] == 'A')].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day,Hour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2010,12,1,8
49,536373,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 09:02:00,2.55,17850.0,United Kingdom,2010,12,1,9
66,536375,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 09:32:00,2.55,17850.0,United Kingdom,2010,12,1,9
100,536378,84519A,TOMATO CHARLIE+LOLA COASTER SET,6,2010-12-01 09:37:00,2.95,14688.0,United Kingdom,2010,12,1,9
120,536381,37444A,YELLOW BREAKFAST CUP AND SAUCER,1,2010-12-01 09:41:00,2.95,15311.0,United Kingdom,2010,12,1,9
172,536385,85049A,TRADITIONAL CHRISTMAS RIBBONS,12,2010-12-01 09:56:00,1.25,17420.0,United Kingdom,2010,12,1,9
203,536389,85014A,BLACK/BLUE POLKADOT UMBRELLA,3,2010-12-01 10:03:00,5.95,12431.0,Australia,2010,12,1,10
220,536390,85123A,WHITE HANGING HEART T-LIGHT HOLDER,64,2010-12-01 10:19:00,2.55,17511.0,United Kingdom,2010,12,1,10
262,536394,85123A,WHITE HANGING HEART T-LIGHT HOLDER,32,2010-12-01 10:39:00,2.55,13408.0,United Kingdom,2010,12,1,10
278,536396,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 10:51:00,2.55,17850.0,United Kingdom,2010,12,1,10


There seems to be no visible pattern yet.\
What about stock codes ending with "B"?

In [41]:
df[df['StockCode'].apply(lambda x: x[-1] == 'B')].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day,Hour
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2010,12,1,8
51,536373,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 09:02:00,2.75,17850.0,United Kingdom,2010,12,1,9
68,536375,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 09:32:00,2.75,17850.0,United Kingdom,2010,12,1,9
90,536378,84997B,RED 3 PIECE RETROSPOT CUTLERY SET,12,2010-12-01 09:37:00,3.75,14688.0,United Kingdom,2010,12,1,9
101,536378,85183B,CHARLIE & LOLA WASTEPAPER BIN FLORA,48,2010-12-01 09:37:00,1.25,14688.0,United Kingdom,2010,12,1,9
102,536378,85071B,RED CHARLIE+LOLA PERSONAL DOORSIGN,96,2010-12-01 09:37:00,0.38,14688.0,United Kingdom,2010,12,1,9
177,536386,85099B,JUMBO BAG RED RETROSPOT,100,2010-12-01 09:57:00,1.65,16029.0,United Kingdom,2010,12,1,9
202,536389,85014B,RED RETROSPOT UMBRELLA,6,2010-12-01 10:03:00,5.95,12431.0,Australia,2010,12,1,10
234,536390,85099B,JUMBO BAG RED RETROSPOT,100,2010-12-01 10:19:00,1.65,17511.0,United Kingdom,2010,12,1,10
280,536396,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 10:51:00,2.75,17850.0,United Kingdom,2010,12,1,10


Again, no pattern.\
The characters probably have no meaning.

Let's create a new column, ClearStock, that contains stock codes without the alphabetic suffix removed to help connect the dots better.

In [42]:
df['ClearStock'] = df['StockCode'].apply(clear_stocks)

In [43]:
df.groupby('ClearStock')['Description'].unique()

ClearStock
10002                          [INFLATABLE POLITICAL GLOBE , nan]
10080                      [GROOVY CACTUS INFLATABLE, nan, check]
10120                                              [DOGGY RUBBER]
10123                                [HEARTS WRAPPING TAPE , nan]
10124           [ARMY CAMO BOOKCOVER TAPE, SPOTS ON RED BOOKCO...
                                      ...                        
gift_0001_20    [Dotcomgiftshop Gift Voucher £20.00, to push o...
gift_0001_30            [Dotcomgiftshop Gift Voucher £30.00, nan]
gift_0001_40                 [Dotcomgiftshop Gift Voucher £40.00]
gift_0001_50                 [Dotcomgiftshop Gift Voucher £50.00]
m                                                        [Manual]
Name: Description, Length: 3423, dtype: object

#### Alphabetic Stock Codes

In [44]:
df_stock_alphabetic = df[df['StockCode'].apply(lambda x: x[0].isalpha())]
print('Rows with alphabetic stock codes:', len(df_stock_alphabetic))
print('Proportion of transactions with alphabetic stock codes:', len(df_stock_alphabetic) / len(df))

Rows with alphabetic stock codes: 2989
Proportion of transactions with alphabetic stock codes: 0.00556983160064177


~0.6% of the transactions contain an alphabetic stock code.

Let's see which descriptions do they point to and how frequently.

In [45]:
df_stock_alphabetic.groupby('StockCode')['Description'].value_counts().sort_values(ascending=False).head(10)

StockCode     Description    
POST          POSTAGE            1252
DOT           DOTCOM POSTAGE      709
M             Manual              566
C2            CARRIAGE            143
D             Discount             77
S             SAMPLES              62
BANK CHARGES  Bank Charges         37
AMAZONFEE     AMAZON FEE           34
CRUK          CRUK Commission      16
DCGSSGIRL     GIRLS PARTY BAG      13
Name: count, dtype: int64

Most alphabetic stock codes represent the descriptions "POSTAGE", "DOTCOM POSTAGE", "Manual" and other descriptions which likely represent accounting adjustments, shipping fees, or internal entries.

---

### Quantity

In [46]:
df['Quantity'].describe()

count    536641.000000
mean          9.620029
std         219.130156
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

The quantity column seems to deviate a lot.

In [47]:
df['Quantity'].quantile(0.95)

np.float64(30.0)

95% of the quantities are less than or equal to 30, which is nowhere near the maximum value (80,955).\
This means the data contains heavy outliers.

The minimum and maximum values are very extreme values and they are the same but with different signs. Let's investigate further:

In [48]:
df[df['Quantity'].isin([80995, -80995])]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day,Hour,ClearStock
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,2011,12,9,9,23843
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,2011,12,9,9,23843


Both transactions have the same item and seem to have been ordered by the same person. Though, one was cancelled.

Let's check if there are transactions with a "0" quantity.

In [49]:
(df['Quantity'] == 0).sum()

np.int64(0)

There are no transactions with a "0" quantity

#### Non-positive Quantity

Let's inspect when quantities are negative and what does that mean.

In [50]:
df_neg_quantity = df[df['Quantity'] < 0]
print('Rows with transactions with a negative quantity:', len(df_neg_quantity))
print('Proportion of transactions with a negative quantity:', len(df_neg_quantity) / len(df))

Rows with transactions with a negative quantity: 10587
Proportion of transactions with a negative quantity: 0.019728272718633127


~2% of the transactions have a negative quantity (10,587 rows).

In [51]:
df_neg_quantity['Quantity'].describe()

count    10587.000000
mean       -45.576367
std       1094.050015
min     -80995.000000
25%        -10.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64

In [52]:
df_neg_quantity['Quantity'].quantile(0.05)

np.float64(-96.0)

Even 95% of negative quantities are more than or equal to -96, which is no where near the lowest negative value (-80,995).

In [53]:
df_neg_quantity['Quantity'].value_counts().sort_index()

Quantity
-80995       1
-74215       1
-9600        2
-9360        1
-9058        1
          ... 
-5         235
-4         501
-3         619
-2        1392
-1        4163
Name: count, Length: 329, dtype: int64

Even when ignoring the fact that the quantities are negative, these values (e.g. -74215) seem unrealistic.

Let's see what is the ratio of transactions that contain negative quantities to be a cancelled transaction.

In [54]:
df_neg_quantity_cancelled = df_neg_quantity[df_neg_quantity['InvoiceNo'].str.startswith('C')]
print('Rows with negative quantites and cancelled transactions:', len(df_neg_quantity_cancelled))
print('Proportion of transactions with negative quantites that are cancelled:', len(df_neg_quantity_cancelled) / len(df_neg_quantity))

Rows with negative quantites and cancelled transactions: 9251
Proportion of transactions with negative quantites that are cancelled: 0.8738074997638613


~87% of transactions with a negative quantity are a cancelled transaction, this suggests a strong correlation between the two occurrences.

Let's see if transactions with a negative quantity are ordered by certain people or not:

In [55]:
df_neg_quantity['CustomerID'].value_counts()

CustomerID
14911.0    226
17841.0    136
17511.0    113
15311.0    112
12607.0    101
          ... 
14087.0      1
12785.0      1
14739.0      1
17673.0      1
16446.0      1
Name: count, Length: 1589, dtype: int64

They are not.

Now, let's see if it has anything to do with negative unit prices.

In [56]:
df_neg_quantity_neg_price = df_neg_quantity[df_neg_quantity['UnitPrice'] < 0]
print('Rows with negative quantites and negative unit price:', len(df_neg_quantity_neg_price))
print('Proportion of transactions with negative quantites that have a negative unit price:', len(df_neg_quantity_neg_price) / len(df_neg_quantity))

Rows with negative quantites and negative unit price: 0
Proportion of transactions with negative quantites that have a negative unit price: 0.0


Transactions with a negative quantity seem to always have non-negative unit prices.\
What about unit prices of zero?

In [57]:
df_neg_quantity_zero_price = df_neg_quantity[df_neg_quantity['UnitPrice'] == 0]
print('Rows with negative quantites and zero unit price:', len(df_neg_quantity_zero_price))
print('Proportion of transactions with negative quantites that have a zero unit price:', len(df_neg_quantity_zero_price) / len(df_neg_quantity))

Rows with negative quantites and zero unit price: 1336
Proportion of transactions with negative quantites that have a zero unit price: 0.12619250023613865


~13% of transactions that have a negative quantity also have a unit price of zero.

---

### Non-positive Unit Prices

In [58]:
df['UnitPrice'].describe()

count    536641.000000
mean          4.632656
std          97.233118
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

Let's see how many negative unit prices are there:

In [59]:
df_neg_price = df[df['UnitPrice'] < 0]
print('Rows with a negative unit price:', len(df_neg_price))
print('Proportion of negative unit prices:', len(df_neg_price) / len(df))

Rows with a negative unit price: 2
Proportion of negative unit prices: 3.7268863169232318e-06


Since there are only 2 transactions with negative unit price, they can be removed without any consequences. Their invoice numbers are also two of the three that start with the letter "A" that we discovered earlier and which should also be removed.

Now, what about unit price of zero?

In [60]:
df_zero_price = df[df['UnitPrice'] == 0]
print('Rows with a unit price of zero:', len(df_zero_price))
print('Proportion of unit prices of zero:', len(df_zero_price) / len(df))

Rows with a unit price of zero: 2510
Proportion of unit prices of zero: 0.0046772423277386555


Only ~0.5% of the transactions contain a unit price of zero.

We've already established that 13% of transactions with a negative quantity also have a unit price of zero. But what about the proportion of transactions with a unit price of zero that also have a negative quantity?

In [61]:
df_zero_price_neg_quantity = df_zero_price[df_zero_price['Quantity'] < 0]
print('Rows with a unit price of zero and a negative quantity:', len(df_zero_price_neg_quantity))
print('Proportion of zero-price transactions with a negative quantity:', len(df_zero_price_neg_quantity) / len(df_zero_price))
print('Proportion of transactions that have both a unit price of zero and a negative quantity:', len(df_zero_price_neg_quantity) / len(df))

Rows with a unit price of zero and a negative quantity: 1336
Proportion of zero-price transactions with a negative quantity: 0.5322709163346614
Proportion of transactions that have both a unit price of zero and a negative quantity: 0.0024895600597047186


~53% of transactions with a "0" unit price also have a negative quantity (1336 rows). Which represents ~0.2% of the total transactions in the dataset.

Let's see if transactions with unit prices of zero are related to cancelled orders:

In [62]:
df_zero_price_cancelled = df_zero_price[df_zero_price['InvoiceNo'].apply(lambda x: x.startswith('C'))]
print('Rows with a unit price of zero that are a cancelled transaction:', len(df_zero_price_cancelled))
print('Proportion of zero-price transactions that are cancelled:', len(df_zero_price_cancelled) / len(df_zero_price))
print('Proportion of rows that have both a unit price and a cancelled transaction:', len(df_zero_price_cancelled) / len(df))

Rows with a unit price of zero that are a cancelled transaction: 0
Proportion of zero-price transactions that are cancelled: 0.0
Proportion of rows that have both a unit price and a cancelled transaction: 0.0


There are no cancelled zero-price transactions.

## Missing Data

It is already known that only the CustomerID and Description columns contain missing values. Now, let's see how many values are missing in each column.

In [63]:
missing_counts = df.isna().sum()
missing_counts

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
Year                0
Month               0
Day                 0
Hour                0
ClearStock          0
dtype: int64

In [64]:
customerid_missing_count = missing_counts.loc['CustomerID']
print('Rows with a missing customer id:', customerid_missing_count)
print('Proportion of transactions with a missing customer id:', customerid_missing_count / len(df))

Rows with a missing customer id: 135037
Proportion of transactions with a missing customer id: 0.2516337737891812


In [65]:
description_missing_count = missing_counts.loc['Description']
print('Rows with a missing description:', description_missing_count)
print('Proportion of transactions with a missing description:', description_missing_count / len(df))

Rows with a missing description: 1454
Proportion of transactions with a missing description: 0.0027094463524031894


~25% of the transactions have a missing customer id (135,037 rows out of 536,641).\
\~0.27% of the transactions have a missing description (1,454 rows out of 536,641).

Let's see if both Description and CustomerID are missing in the same rows.

In [66]:
# Getting the length of the dataframe where both the description and customer id are missing
len(df[df['Description'].isna() & df['CustomerID'].isna()])

1454

Since the number of rows where both the description and customer id are missing is equal to the number of rows where the description is missing, this means that whenever the description is missing, the customer id is also missing. This also implies that in all incomplete rows, CustomerID is missing.

In [67]:
# Creating a dataframe with only rows with missing values and another with only rows with non-missing values
df_missing = df[df.isna().any(axis=1)]
df_notmissing = df.dropna()

### Missings Customer IDs

Since all rows with missing values have a missing customer id, the dataframe "df_missing" contains all rows with missing customer ids.

In [68]:
# Getting the value counts of countries that have a missing customer id in the same row:
missing_country = df_missing['Country'].value_counts()

# Getting the proportions of each country's occurrences with a missing customer id:
missing_country_percent = (df_missing['Country'].value_counts() / df['Country'].value_counts()).dropna()

# Concatenating both series in one dataframe for better visualization:
pd.concat([missing_country, missing_country_percent], axis=1)

,count,count
Country,,
United Kingdom,133572,0.272429
EIRE,709,0.086632
Hong Kong,284,1.000000
Unspecified,201,0.454751
Switzerland,117,0.058676
France,66,0.007727
Israel,47,0.159864
Portugal,39,0.025828
Bahrain,2,0.105263


Countries with the highest rate of missing customer id:\
100% of the transactions from Hong Kong don't have a customer ID (284 rows).\
45% of the transactions that have an unspecified country don't have a customer ID (201 rows).\
27% of the transactions from the UK don't have a customer ID. (133,572 rows).

Next up let's check for the correlation between a missing customer id and timestamp features.
Starting with the year:

In [69]:
(df_missing['Year'].value_counts() / df['Year'].value_counts()).sort_values(ascending=False)

Year
2010    0.372240
2011    0.241398
Name: count, dtype: float64

24% of the transactions in the year 2011 don't have a customer ID.\
37% of the transactions in the year 2010 don't have a customer ID.\
No clear pattern or correlation.

What about months?

In [70]:
(df_missing['Month'].value_counts() / df['Month'].value_counts()).sort_values(ascending=False)

Month
1     0.379083
12    0.349428
7     0.305880
2     0.267149
6     0.246743
3     0.244875
11    0.229305
4     0.226019
5     0.220787
8     0.217317
9     0.188564
10    0.167437
Name: count, dtype: float64

38% of the transactions in January don't have a customer ID.\
35% of the transactions in December don't have a customer ID.\
31% of the transactions in July don't have a customer ID.\
There seems to be no clear patterns regarding missing customer ids and timestamp features.

Let's see if transactions that have a missing customer id may have a present customer id in another transaction within the same invoice.

In [71]:
# Checking if there are invoice numbers in df_missing that are also present in df_notmissing
len(df_missing[df_missing['InvoiceNo'].isin(df_notmissing['InvoiceNo'])])

0

When a customer id is missing, no other customer id is present within the same invoice.

### Missing Descriptions

In [72]:
# Creating a dataframe with only rows that have a missing description:
df_missing_desc = df_missing[df_missing['Description'].isna()]

Let's see if there is a correlation between missing descriptions and zero-price transactions:

In [73]:
df_missing_desc_zero_price = df_missing_desc[df_missing_desc['UnitPrice'] == 0]
print('Rows with missing description that also have a unit price of zero:', len(df_missing_desc_zero_price))
print('Proportion of transactions with a missing description that also have a unit price of zero:', len(df_missing_desc_zero_price) / len(df_missing_desc))

Rows with missing description that also have a unit price of zero: 1454
Proportion of transactions with a missing description that also have a unit price of zero: 1.0


100% of the transaction with a missing description have the UnitPrice set to 0. This could either mean that the item was given for free due to an offer, or that this is simply an error in data entry.\
Let's see if all 0 unit prices only happen when the description is missing:

In [74]:
len(df_zero_price), len(df_missing_desc)

(2510, 1454)

Since the number of transactions with a unit price of zero is larger than the number of transactions with a missing description, transactions with a unit price of zero don't necessarily mean a missing description.

Now, we'd check negative unit prices too, but we know that there are only 2 rows with a negative unit price and they will be removed anyway. Therefore, checking the correlation between missing descriptions and negative unit prices won't be of use.

Let's see if the stock codes of missing descriptions have a non-missing description in other transactions.

In [75]:
# Checking if there are stock codes in both df_missing_desc and df_notmissing
df_special_stocks = df_notmissing[df_notmissing['StockCode'].isin(df_missing_desc['StockCode'])]
print('Unique stock codes that are present in both df_missing_desc and df_notmissing:', len(df_special_stocks['StockCode'].unique()))
print('Proportion of rows with a missing description containing those unique stock codes:', len(df_special_stocks['StockCode'].unique()) / len(df_missing_desc))

Unique stock codes that are present in both df_missing_desc and df_notmissing: 822
Proportion of rows with a missing description containing those unique stock codes: 0.5653370013755158


There are 822 stock codes that have the potential to fill ~57% of the missing descriptions during the data cleaning process.

---

# TDL:

TDL in data cleaning:
- Deal with duplicates
- Turn the data in InvoiceDate to datetime64
- Deal with rows with InvoiceNo starting with "A"
- Remove the "C" in the cancelled invoice numbers then add a new column to mark cancelled orders
- Deal with rows with unspecified countries
- Deal with rows with negative quantity
- Deal with rows with a negative unit price
- Deal with rows with unit price of zero
- Decide what to do with missing values in the CustomerID and Description columns
- See if missing values in the Description column can be replaced by stock codes